In [1]:
import os
import cv2
import json
import numpy as np
from collections import defaultdict, deque

In [2]:
class MemoryCameraMatcher:
    def __init__(self, history_frames=5):
        self.camera_pairs = [
            ('FrontLeft', 'Front'),
            ('Front', 'FrontRight'),
            ('FrontRight', 'BackRight'),
            ('BackRight', 'Back'),
            ('Back', 'BackLeft'),
            ('BackLeft', 'FrontLeft')
        ]
        
        # Reduced overlap regions to make matching more strict
        self.overlap_regions = {
            ('FrontLeft', 'Front'): {'left': (0.7, 1.0), 'right': (0.0, 0.3)},
            ('Front', 'FrontRight'): {'left': (0.7, 1.0), 'right': (0.0, 0.3)},
            ('FrontRight', 'BackRight'): {'left': (0.7, 1.0), 'right': (0.0, 0.3)},
            ('BackRight', 'Back'): {'left': (0.7, 1.0), 'right': (0.0, 0.3)},
            ('Back', 'BackLeft'): {'left': (0.7, 1.0), 'right': (0.0, 0.3)},
            ('BackLeft', 'FrontLeft'): {'left': (0.7, 1.0), 'right': (0.0, 0.3)}
        }
        
        # Vehicle type priority for hierarchical matching
        self.vehicle_priority = {
            'truck': 1,
            'bus': 2,
            'car': 3,
            'motorcycle': 4,
            'bicycle': 5
        }
        
        # Match history for each camera pair
        self.match_history = defaultdict(lambda: defaultdict(lambda: deque(maxlen=history_frames)))
        self.history_frames = history_frames

    def calculate_feature_similarity(self, feat1, feat2):
        """Calculate similarity between enhanced feature sets"""
        if feat1 is None or feat2 is None:
            return 0.0

        try:
            # Compare histograms for each region and channel
            total_score = 0
            processed_channels = 0
            weight = 1.0 / 6  # Equal weight for each histogram comparison

            for region in ['upper', 'lower']:
                for channel in ['h', 's', 'v']:
                    key = f'{region}_{channel}'

                    # Check if key exists in both features
                    if key not in feat1 or key not in feat2:
                        continue

                    try:
                        # Convert lists to numpy arrays and ensure float32 type
                        hist1 = np.array(feat1[key], dtype=np.float32).reshape(-1, 1)
                        hist2 = np.array(feat2[key], dtype=np.float32).reshape(-1, 1)

                        if hist1.size > 0 and hist2.size > 0:
                            score = cv2.compareHist(hist1, hist2, cv2.HISTCMP_CORREL)
                            total_score += weight * max(0, score)
                            processed_channels += 1
                    except ValueError as ve:
                        print(f"Error processing histogram for {key}: {str(ve)}")
                        continue

            # Normalize score based on number of successfully processed channels
            if processed_channels > 0:
                total_score = total_score * (6 / processed_channels)  # Normalize to original scale

            # Compare dominant colors
            if 'dominant_colors' in feat1 and 'dominant_colors' in feat2:
                color_sim = self.compare_dominant_colors(
                    feat1['dominant_colors'],
                    feat2['dominant_colors']
                )
                total_score = 0.7 * total_score + 0.3 * color_sim

            return total_score

        except Exception as e:
            print(f"Error calculating feature similarity: {str(e)}")
            print(f"Feature keys in feat1: {feat1.keys() if isinstance(feat1, dict) else 'Not a dict'}")
            print(f"Feature keys in feat2: {feat2.keys() if isinstance(feat2, dict) else 'Not a dict'}")
            return 0.0
    def compare_dominant_colors(self, colors1, colors2):
        """Compare dominant color sets considering percentages"""
        if not colors1 or not colors2:
            return 0
        
        total_sim = 0
        for c1 in colors1:
            color1 = np.array(c1['color'])
            pct1 = c1['percentage']
            
            max_color_sim = 0
            for c2 in colors2:
                color2 = np.array(c2['color'])
                pct2 = c2['percentage']
                
                # Calculate color similarity in HSV space
                color_dist = np.exp(-np.sum(np.abs(color1 - color2)) / 255.0)
                # Weight by both percentages
                sim = color_dist * min(pct1, pct2)
                max_color_sim = max(max_color_sim, sim)
            
            total_sim += max_color_sim
        
        return total_sim / len(colors1)

    def is_in_overlap_region(self, center_x, image_width, camera_pair, is_left_camera):
        """Check if object's center is in the overlap region"""
        relative_x = center_x / image_width
        overlap = self.overlap_regions[camera_pair]
        region = overlap['left'] if is_left_camera else overlap['right']
        tolerance = 0.05  # 5% tolerance for border cases
        return region[0] - tolerance <= relative_x <= region[1] + tolerance

    def get_vehicle_priority(self, obj_type):
        """Get priority for hierarchical matching"""
        # Remove common prefixes/suffixes to get base vehicle type
        base_type = obj_type.lower().replace('vehicle_', '').replace('_', ' ').strip()
        return self.vehicle_priority.get(base_type, 10)  # Default low priority if type unknown

    def get_consistent_match(self, obj1_id, obj2_id, camera_pair):
        """Check if this match has been consistent in recent frames"""
        history = self.match_history[camera_pair][obj1_id]
        if not history:
            return True
        
        match_count = sum(1 for match in history if match == obj2_id)
        consistency_threshold = self.history_frames * 0.6
        
        return match_count >= consistency_threshold

    def match_objects(self, frame_file, frame_num, root_dir, scenario_folder):
        """Match objects between adjacent cameras using enhanced features"""
        matches = []
        
        for cam1, cam2 in self.camera_pairs:
            cam1_path = f"Camera_{cam1}"
            cam2_path = f"Camera_{cam2}"
            camera_pair = (cam1, cam2)
            
            # Get paths
            json1_path = os.path.join(root_dir, cam1_path, scenario_folder, 
                                    'Annotations_JSON', frame_file.replace('.jpg', '.json'))
            json2_path = os.path.join(root_dir, cam2_path, scenario_folder, 
                                    'Annotations_JSON', frame_file.replace('.jpg', '.json'))
            
            if not all(os.path.exists(p) for p in [json1_path, json2_path]):
                continue
            
            try:
                # Load annotations
                with open(json1_path, 'r') as f1, open(json2_path, 'r') as f2:
                    data1 = json.load(f1)
                    data2 = json.load(f2)
                
                image_width1 = data1['images'][0]['width']
                image_width2 = data2['images'][0]['width']
                
                # Group objects by type for hierarchical matching
                objects1 = defaultdict(list)
                objects2 = defaultdict(list)
                
                # Process annotations and group by vehicle type
                for ann in data1['annotations']:
                    center_x = ann['attributes']['center'][0]
                    if self.is_in_overlap_region(center_x, image_width1, camera_pair, True):
                        obj_type = ann['attributes']['class']
                        priority = self.get_vehicle_priority(obj_type)
                        objects1[priority].append({
                            'id': ann['track_id'],
                            'type': obj_type,
                            'features': ann['attributes'].get('features'),
                            'center': ann['attributes']['center'],
                            'confidence': ann['attributes']['confidence']
                        })
                
                for ann in data2['annotations']:
                    center_x = ann['attributes']['center'][0]
                    if self.is_in_overlap_region(center_x, image_width2, camera_pair, False):
                        obj_type = ann['attributes']['class']
                        priority = self.get_vehicle_priority(obj_type)
                        objects2[priority].append({
                            'id': ann['track_id'],
                            'type': obj_type,
                            'features': ann['attributes'].get('features'),
                            'center': ann['attributes']['center'],
                            'confidence': ann['attributes']['confidence']
                        })
                
                # Process objects in priority order (larger vehicles first)
                matched_ids2 = set()
                for priority in sorted(set(objects1.keys()) | set(objects2.keys())):
                    for obj1 in objects1.get(priority, []):
                        best_match = None
                        best_score = 0.5  # Minimum threshold for matching
                        
                        for obj2 in objects2.get(priority, []):
                            if obj2['id'] in matched_ids2:
                                continue
                            
                            # Type checking
                            if not (obj1['type'] == obj2['type']):
                                continue
                            
                            # Calculate feature similarity
                            feature_sim = self.calculate_feature_similarity(
                                obj1['features'],
                                obj2['features']
                            )
                            
                            # Check history for consistent matches
                            history_bonus = 0.2 if self.get_consistent_match(
                                obj1['id'],
                                obj2['id'],
                                camera_pair
                            ) else 0
                            
                            # Combine scores
                            total_score = feature_sim + history_bonus
                            
                            if total_score > best_score:
                                best_score = total_score
                                best_match = obj2
                        
                        if best_match:
                            matched_ids2.add(best_match['id'])
                            match_str = f"{obj1['id']} = {best_match['id']}"
                            
                            # Update match history
                            self.match_history[camera_pair][obj1['id']].append(best_match['id'])
                            
                            if match_str not in matches and match_str[::-1] not in matches:
                                matches.append(match_str)
            
            except Exception as e:
                print(f"Error processing cameras {cam1}-{cam2} for frame {frame_file}: {str(e)}")
                continue
        
        return matches



In [3]:
def process_scenario(root_dir, scenario_folder):
    """Process all frames in a scenario"""
    matcher = MemoryCameraMatcher(history_frames=5)
    
    camera_path = os.path.join(root_dir, 'Camera_Back', scenario_folder)
    frame_files = [f for f in os.listdir(camera_path) 
                  if f.endswith('.jpg') and not 'Segmented_Data' in f]
    frame_files.sort()
    
    print(f"Processing {len(frame_files)} frames in {scenario_folder}")
    
    for frame_num, frame_file in enumerate(frame_files):
        print(f"\nFrame {frame_file}:")
        matches = matcher.match_objects(frame_file, frame_num, root_dir, scenario_folder)
        for match in matches:
            print(match)


    

In [4]:
root_dir = "type1_subtype1_accident/ego_vehicle/"
scenario_folder = "Town05_type001_subtype0001_scenario00007"
process_scenario(root_dir, scenario_folder)


Processing 53 frames in Town05_type001_subtype0001_scenario00007

Frame Town05_type001_subtype0001_scenario00007_001.jpg:

Frame Town05_type001_subtype0001_scenario00007_002.jpg:
Veh_FL_A = Veh_FF_A

Frame Town05_type001_subtype0001_scenario00007_003.jpg:
Veh_FL_A = Veh_FF_A

Frame Town05_type001_subtype0001_scenario00007_004.jpg:
Veh_FL_A = Veh_FF_A

Frame Town05_type001_subtype0001_scenario00007_005.jpg:
Veh_FL_A = Veh_FF_A
Veh_FF_D = Veh_FR_A

Frame Town05_type001_subtype0001_scenario00007_006.jpg:
Veh_FL_A = Veh_FF_A

Frame Town05_type001_subtype0001_scenario00007_007.jpg:
Veh_FL_A = Veh_FF_A

Frame Town05_type001_subtype0001_scenario00007_008.jpg:
Veh_FF_D = Veh_FR_A
Veh_BB_C = Veh_BL_B

Frame Town05_type001_subtype0001_scenario00007_009.jpg:
Veh_BB_C = Veh_BL_B

Frame Town05_type001_subtype0001_scenario00007_010.jpg:
Veh_BB_C = Veh_BL_B

Frame Town05_type001_subtype0001_scenario00007_011.jpg:
Veh_BB_B = Veh_BL_A

Frame Town05_type001_subtype0001_scenario00007_012.jpg:
Veh_FL_B = 